# Verify Postgres Migration: Home Assistant RAG

`02_verify_refactor.ipynb` confirmed the modularized `ingestion/`/`app/` code worked with in-memory `minsearch`. Since then, both modules were migrated to Postgres + pgvector, adding lexical (full-text) search alongside semantic (vector) search, combined via Reciprocal Rank Fusion into hybrid search.

This notebook confirms the migration works and compares the three retrieval modes (`vector`, `text`, `hybrid`) side by side on a few questions.

> **Snapshot notice**: this notebook is a point-in-time snapshot of the dev process, not a maintained test suite - see the same notice in `02_verify_refactor.ipynb` for the exact reason. If it doesn't run against your current checkout, `git checkout` the commit it was added in first:
> ```bash
> git log --oneline -- notebooks/03_verify_postgres_migration.ipynb
> git checkout <that-commit-hash>
> ```

**Prerequisite**: Postgres must be running (`docker compose up -d postgres`) and ingestion must have been run (`uv run python ingestion/ingest.py`) before this notebook will work.

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

from app import rag

## 1. Sanity-check the connection + data

If this fails, either Postgres isn't running, or `ingestion/ingest.py` hasn't
been run yet against it.

In [2]:
import psycopg

with psycopg.connect(rag.get_pg_dsn()) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT source, count(*) FROM documents GROUP BY source;")
        for row in cur.fetchall():
            print(row)

('esphome', 66)
('zigbee2mqtt', 64)
('home_assistant', 356)


## 2. Compare retrieval modes on the same question

Not a formal evaluation (that's a later commit with a proper golden set + hit-rate/MRR). This is just an eyeball check that hybrid returns sensible results and differs meaningfully from vector-only / text-only.

In [3]:
question = "How do I pair a new Zigbee device?"

for mode in ["vector", "text", "hybrid"]:
    chunks = rag.retrieve(question, mode=mode, top_k=3)
    print(f"--- mode={mode} ---")
    for c in chunks:
        print(f"  [{c['source']}] {c['title']} — {c['content'][:80]}...")
    print()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

--- mode=vector ---
  [zigbee2mqtt] Getting started | Zigbee2MQTT —  start controlling it using the frontend and MQTT messages.
Zigbee2MQTT:info  20...
  [zigbee2mqtt] Allowing devices to join | Zigbee2MQTT — long-term). However, this can be useful for devices that are awkward in router s...
  [zigbee2mqtt] Getting started | Zigbee2MQTT — Getting started
Prerequisites
In order to use Zigbee2MQTT we need the following ...

--- mode=text ---
  [zigbee2mqtt] MQTT Topics and Messages | Zigbee2MQTT — ove a device add the optional
force
property (default
false
) to the payload, ex...

--- mode=hybrid ---
  [zigbee2mqtt] MQTT Topics and Messages | Zigbee2MQTT — ove a device add the optional
force
property (default
false
) to the payload, ex...
  [zigbee2mqtt] Getting started | Zigbee2MQTT —  start controlling it using the frontend and MQTT messages.
Zigbee2MQTT:info  20...
  [zigbee2mqtt] Allowing devices to join | Zigbee2MQTT — long-term). However, this can be useful for devices that are awk

## 3. Full smoke test (retrieval + LLM answer)

Same two questions used in earlier notebooks. The answers should look just as grounded/relevant now that retrieval is hybrid instead of vector-only.

In [4]:
for q in [
    "How do I template a sensor value in Home Assistant?",
    "How do I pair a new Zigbee device?",
]:
    result = rag.answer_question(q)
    print(f"Q: {result['question']}\n")
    print(f"A: {result['answer']}\n")
    print("Sources:", [s["url"] for s in result["sources"][:3]])
    print("-" * 100)

Q: How do I template a sensor value in Home Assistant?

A: **Creating a template sensor in Home Assistant**

A template sensor lets you derive a new state (or attributes) from other entities in Home Assistant.  
You can create it either through the UI (Helpers → “Add template sensor”) or directly in YAML.

---

### 1. State‑based template sensor (updates automatically)

Add the following to your `configuration.yaml` (or via the UI):

```yaml
template:
  - sensor:
      - name: "Average temperature"
        unit_of_measurement: "°C"
        state: >
          {% set bedroom = states('sensor.bedroom_temperature') | float %}
          {% set kitchen = states('sensor.kitchen_temperature') | float %}
          {{ ((bedroom + kitchen) / 2) | round(1, default=0) }}
```

* `name` – the entity name that will appear in the UI.  
* `unit_of_measurement` – optional, shows the unit.  
* `state` – the Jinja2 template that calculates the value.  
  The sensor will automatically refresh whenever any r